# Hybrid Soil Analysis System

## Objective

This notebook combines information from:

1. Soil image analysis using the fine-tuned EfficientNetB0 model.
2. Structured soil-data analysis using Gradient Boosting models for
   nitrogen, phosphorus, and potassium deficiency prediction.

The hybrid system combines these outputs to provide a consolidated
soil assessment containing:
- predicted soil type,
- nitrogen status,
- phosphorus status,
- potassium status,
- prediction confidence,
- soil-health score,
- overall soil assessment.

The hybrid analysis demonstrates how image-based and structured-data
models can be used together to support intelligent soil analysis.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")
print("TensorFlow version:", tf.__version__)

Libraries loaded successfully.
TensorFlow version: 2.20.0


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
MODEL_DIR = "/content/drive/MyDrive/Soil_Analytics_Project/Models"

IMAGE_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "efficientnet_soil_classifier_finetuned.keras"
)

N_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "N_deficiency_gradient_boosting.pkl"
)

P_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "P_deficiency_gradient_boosting.pkl"
)

K_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "K_deficiency_gradient_boosting.pkl"
)

image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH)

model_N = joblib.load(N_MODEL_PATH)
model_P = joblib.load(P_MODEL_PATH)
model_K = joblib.load(K_MODEL_PATH)

print("All models loaded successfully.")

All models loaded successfully.


In [8]:
class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

print("Classes loaded:")
for i, name in enumerate(class_names):
    print(i, name)

Classes loaded:
0 Alluvial_Soil
1 Arid_Soil
2 Black_Soil
3 Laterite_Soil
4 Mountain_Soil
5 Red_Soil
6 Yellow_Soil


In [9]:
IMAGE_PATH = "/content/drive/MyDrive/Soil_Analytics_Project/Data/Processed/Image_Split/test/Alluvial_Soil/2.jpg"

image = tf.keras.utils.load_img(
    IMAGE_PATH,
    target_size=(224, 224)
)

image_array = tf.keras.utils.img_to_array(image)
image_array = np.expand_dims(image_array, axis=0)

image_prediction = image_model.predict(
    image_array,
    verbose=0
)

predicted_image_class = np.argmax(image_prediction[0])
image_confidence = image_prediction[0][predicted_image_class]

print("Predicted soil class:", class_names[predicted_image_class])
print("Image confidence:", image_confidence)

Predicted soil class: Alluvial_Soil
Image confidence: 0.72998387


In [13]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Soil_Analytics_Project/Data/Raw/Structured_Data/Crop_recommendationV2.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (2200, 23)


In [14]:
feature_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall",
    "soil_moisture",
    "soil_type",
    "sunlight_exposure",
    "wind_speed",
    "co2_concentration",
    "organic_matter",
    "irrigation_frequency",
    "crop_density",
    "pest_pressure",
    "fertilizer_usage",
    "growth_stage",
    "urban_area_proximity",
    "water_source_type",
    "frost_risk",
    "water_usage_efficiency"
]

print("Number of features:", len(feature_columns))

Number of features: 22


In [15]:
sample_index = 0

structured_sample = df.iloc[[sample_index]]

print("Selected sample index:", sample_index)
print("Structured sample shape:", structured_sample.shape)

display(structured_sample)

Selected sample index: 0
Structured sample shape: (1, 23)


,N,P,K,temperature,humidity,ph,rainfall,label,soil_moisture,soil_type,...,organic_matter,irrigation_frequency,crop_density,pest_pressure,fertilizer_usage,growth_stage,urban_area_proximity,water_source_type,frost_risk,water_usage_efficiency
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice,29.446064,2,...,3.121395,4,11.74391,57.607308,188.194958,1,2.719614,3,95.649985,1.193293


## Hybrid Combination Logic

The hybrid system uses late-stage result integration.

The image model and structured models predict different but complementary
outputs:

- The image model predicts the soil type.
- The structured models predict N, P, and K deficiency status.

Because these outputs represent different prediction targets, their
probabilities are not directly averaged or mathematically fused.

Instead, the system consolidates the outputs into a single assessment
containing soil type, nutrient status, prediction confidence, and a
project-defined nutrient-status score.

In [16]:
N_input = structured_sample[
    [col for col in feature_columns if col != "N"]
]

P_input = structured_sample[
    [col for col in feature_columns if col != "P"]
]

K_input = structured_sample[
    [col for col in feature_columns if col != "K"]
]

N_prediction = model_N.predict(N_input)[0]
P_prediction = model_P.predict(P_input)[0]
K_prediction = model_K.predict(K_input)[0]

N_probability = model_N.predict_proba(N_input)[0]
P_probability = model_P.predict_proba(P_input)[0]
K_probability = model_K.predict_proba(K_input)[0]

print("N prediction:", N_prediction)
print("N probabilities:", N_probability)

print("\nP prediction:", P_prediction)
print("P probabilities:", P_probability)

print("\nK prediction:", K_prediction)
print("K probabilities:", K_probability)

N prediction: 0
N probabilities: [0.99521019 0.00478981]

P prediction: 0
P probabilities: [0.96753113 0.03246887]

K prediction: 0
K probabilities: [0.98495473 0.01504527]


In [20]:
# Convert nutrient predictions into readable labels
nutrient_status = {
    "N": "Deficient" if N_prediction == 1 else "Not Deficient",
    "P": "Deficient" if P_prediction == 1 else "Not Deficient",
    "K": "Deficient" if K_prediction == 1 else "Not Deficient"
}

# Combine image and structured-model outputs
hybrid_result = {
    "Soil Type": class_names[predicted_image_class],
    "Image Confidence": round(float(image_confidence), 4),

    "N Status": nutrient_status["N"],
    "N Confidence": round(float(N_probability[N_prediction]), 4),

    "P Status": nutrient_status["P"],
    "P Confidence": round(float(P_probability[P_prediction]), 4),

    "K Status": nutrient_status["K"],
    "K Confidence": round(float(K_probability[K_prediction]), 4)
}

print("Hybrid Soil Analysis Result")
print("-" * 40)

for key, value in hybrid_result.items():
    print(f"{key}: {value}")

Hybrid Soil Analysis Result
----------------------------------------
Soil Type: Alluvial_Soil
Image Confidence: 0.73
N Status: Not Deficient
N Confidence: 0.9952
P Status: Not Deficient
P Confidence: 0.9675
K Status: Not Deficient
K Confidence: 0.985


In [30]:
def confidence_level(confidence):
    if confidence >= 0.70:
        return "High"
    elif confidence >= 0.50:
        return "Moderate"
    else:
        return "Low"

### Confidence Interpretation

Prediction probabilities are reported for every model output. For
demonstration purposes, probabilities of 70% or above are described as
"High", 50–69.99% as "Moderate", and below 50% as "Low".

These categories are project-defined interpretation thresholds and should not
be interpreted as calibrated clinical, laboratory, or agronomic confidence
levels.

In [21]:
# Assign 1 point for each nutrient predicted as not deficient
nutrient_scores = {
    "N": 1 if N_prediction == 0 else 0,
    "P": 1 if P_prediction == 0 else 0,
    "K": 1 if K_prediction == 0 else 0
}

soil_health_score = (
    sum(nutrient_scores.values()) / 3
) * 100

print("Nutrient scores:", nutrient_scores)
print("Soil health score:", round(soil_health_score, 2), "/ 100")

Nutrient scores: {'N': 1, 'P': 1, 'K': 1}
Soil health score: 100.0 / 100


In [22]:
if soil_health_score == 100:
    overall_assessment = "No modeled N, P, or K deficiency detected."
elif soil_health_score >= 66.67:
    overall_assessment = "One modeled nutrient deficiency detected."
elif soil_health_score >= 33.33:
    overall_assessment = "Two modeled nutrient deficiencies detected."
else:
    overall_assessment = "Modeled N, P, and K deficiencies detected."

print("Overall assessment:", overall_assessment)

Overall assessment: No modeled N, P, or K deficiency detected.


## End-to-End Hybrid Soil Analysis Function

This function combines the image classification model with the
structured-data nutrient deficiency models.

Input:
- Soil image
- Corresponding structured soil measurements

Output:
- Soil type and image confidence
- N, P, and K deficiency status
- Nutrient prediction confidence
- Project-defined soil-health score
- Overall modeled assessment

In [23]:
def hybrid_soil_analysis(image_path, structured_sample):
    """
    Perform combined soil image and structured-data analysis.

    Parameters
    ----------
    image_path : str
        Path to the soil image.

    structured_sample : pandas.DataFrame
        One-row DataFrame containing the structured soil measurements.

    Returns
    -------
    dict
        Combined hybrid soil analysis result.
    """

    # --------------------------------------------------
    # 1. IMAGE PREDICTION
    # --------------------------------------------------

    image = tf.keras.utils.load_img(
        image_path,
        target_size=(224, 224)
    )

    image_array = tf.keras.utils.img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)

    image_prediction = image_model.predict(
        image_array,
        verbose=0
    )

    predicted_image_class = np.argmax(image_prediction[0])
    image_confidence = float(
        image_prediction[0][predicted_image_class]
    )

    # --------------------------------------------------
    # 2. STRUCTURED-DATA INPUTS
    # --------------------------------------------------

    N_input = structured_sample[
        [col for col in feature_columns if col != "N"]
    ]

    P_input = structured_sample[
        [col for col in feature_columns if col != "P"]
    ]

    K_input = structured_sample[
        [col for col in feature_columns if col != "K"]
    ]

    # --------------------------------------------------
    # 3. N/P/K PREDICTIONS
    # --------------------------------------------------

    N_prediction = model_N.predict(N_input)[0]
    P_prediction = model_P.predict(P_input)[0]
    K_prediction = model_K.predict(K_input)[0]

    N_probability = model_N.predict_proba(N_input)[0]
    P_probability = model_P.predict_proba(P_input)[0]
    K_probability = model_K.predict_proba(K_input)[0]

    # --------------------------------------------------
    # 4. CONVERT TO READABLE STATUS
    # --------------------------------------------------

    N_status = "Deficient" if N_prediction == 1 else "Not Deficient"
    P_status = "Deficient" if P_prediction == 1 else "Not Deficient"
    K_status = "Deficient" if K_prediction == 1 else "Not Deficient"

    # --------------------------------------------------
    # 5. SOIL-HEALTH SCORE
    # --------------------------------------------------

    nutrient_scores = {
        "N": 1 if N_prediction == 0 else 0,
        "P": 1 if P_prediction == 0 else 0,
        "K": 1 if K_prediction == 0 else 0
    }

    soil_health_score = (
        sum(nutrient_scores.values()) / 3
    ) * 100

    # --------------------------------------------------
    # 6. OVERALL ASSESSMENT
    # --------------------------------------------------

    if soil_health_score == 100:
        overall_assessment = (
            "No modeled N, P, or K deficiency detected."
        )
    elif soil_health_score >= 66.67:
        overall_assessment = (
            "One modeled nutrient deficiency detected."
        )
    elif soil_health_score >= 33.33:
        overall_assessment = (
            "Two modeled nutrient deficiencies detected."
        )
    else:
        overall_assessment = (
            "Modeled N, P, and K deficiencies detected."
        )

    # --------------------------------------------------
    # 7. FINAL RESULT
    # --------------------------------------------------

    result = {
        "Soil Type": class_names[predicted_image_class],
        "Image Confidence": round(image_confidence, 4),

        "N Status": N_status,
        "N Confidence": round(
            float(N_probability[N_prediction]), 4
        ),

        "P Status": P_status,
        "P Confidence": round(
            float(P_probability[P_prediction]), 4
        ),

        "K Status": K_status,
        "K Confidence": round(
            float(K_probability[K_prediction]), 4
        ),

        "Soil Health Score": round(
            soil_health_score, 2
        ),

        "Overall Assessment": overall_assessment
    }

    return result

In [24]:
hybrid_output = hybrid_soil_analysis(
    IMAGE_PATH,
    structured_sample
)

print("===== HYBRID SOIL ANALYSIS =====")

for key, value in hybrid_output.items():
    print(f"{key}: {value}")

===== HYBRID SOIL ANALYSIS =====
Soil Type: Alluvial_Soil
Image Confidence: 0.73
N Status: Not Deficient
N Confidence: 0.9952
P Status: Not Deficient
P Confidence: 0.9675
K Status: Not Deficient
K Confidence: 0.985
Soil Health Score: 100.0
Overall Assessment: No modeled N, P, or K deficiency detected.


## Multiple-Sample Hybrid Testing

The hybrid analysis function is tested on multiple structured soil
records to verify that it produces consistent outputs across different
inputs.

The image input is kept fixed for this pipeline demonstration because
the current dataset does not provide verified image-to-structured-record
pairings.

In [25]:
test_indices = [0, 1, 2, 3, 4]

hybrid_results = []

for idx in test_indices:
    sample = df.iloc[[idx]]

    result = hybrid_soil_analysis(
        IMAGE_PATH,
        sample
    )

    result["Sample Index"] = idx

    hybrid_results.append(result)

hybrid_results_df = pd.DataFrame(hybrid_results)

hybrid_results_df = hybrid_results_df[
    [
        "Sample Index",
        "Soil Type",
        "Image Confidence",
        "N Status",
        "N Confidence",
        "P Status",
        "P Confidence",
        "K Status",
        "K Confidence",
        "Soil Health Score",
        "Overall Assessment"
    ]
]

display(hybrid_results_df)

,Sample Index,Soil Type,Image Confidence,N Status,N Confidence,P Status,P Confidence,K Status,K Confidence,Soil Health Score,Overall Assessment
0,0,Alluvial_Soil,0.73,Not Deficient,0.9952,Not Deficient,0.9675,Not Deficient,0.9850,100.0,"No modeled N, P, or K deficiency detected."
1,1,Alluvial_Soil,0.73,Not Deficient,0.9825,Not Deficient,0.9834,Not Deficient,0.9834,100.0,"No modeled N, P, or K deficiency detected."
2,2,Alluvial_Soil,0.73,Not Deficient,0.9921,Not Deficient,0.9778,Not Deficient,0.9869,100.0,"No modeled N, P, or K deficiency detected."
3,3,Alluvial_Soil,0.73,Not Deficient,0.9868,Not Deficient,0.9856,Not Deficient,0.9892,100.0,"No modeled N, P, or K deficiency detected."
4,4,Alluvial_Soil,0.73,Not Deficient,0.9920,Not Deficient,0.9845,Not Deficient,0.9767,100.0,"No modeled N, P, or K deficiency detected."


In [26]:
# Select samples from different parts of the dataset
test_indices = [0, 500, 1000, 1500, 2000]

hybrid_results = []

for idx in test_indices:
    sample = df.iloc[[idx]]

    result = hybrid_soil_analysis(
        IMAGE_PATH,
        sample
    )

    result["Sample Index"] = idx

    hybrid_results.append(result)

hybrid_results_df = pd.DataFrame(hybrid_results)

hybrid_results_df = hybrid_results_df[
    [
        "Sample Index",
        "Soil Type",
        "Image Confidence",
        "N Status",
        "N Confidence",
        "P Status",
        "P Confidence",
        "K Status",
        "K Confidence",
        "Soil Health Score",
        "Overall Assessment"
    ]
]

display(hybrid_results_df)

,Sample Index,Soil Type,Image Confidence,N Status,N Confidence,P Status,P Confidence,K Status,K Confidence,Soil Health Score,Overall Assessment
0,0,Alluvial_Soil,0.73,Not Deficient,0.9952,Not Deficient,0.9675,Not Deficient,0.9850,100.00,"No modeled N, P, or K deficiency detected."
1,500,Alluvial_Soil,0.73,Deficient,0.9634,Not Deficient,0.9906,Deficient,0.9114,33.33,Two modeled nutrient deficiencies detected.
2,1000,Alluvial_Soil,0.73,Not Deficient,0.9805,Not Deficient,0.9332,Not Deficient,0.9640,100.00,"No modeled N, P, or K deficiency detected."
3,1500,Alluvial_Soil,0.73,Not Deficient,0.7936,Not Deficient,0.9796,Not Deficient,0.9925,100.00,"No modeled N, P, or K deficiency detected."
4,2000,Alluvial_Soil,0.73,Not Deficient,0.9852,Not Deficient,0.9721,Not Deficient,0.9603,100.00,"No modeled N, P, or K deficiency detected."


In [27]:
def display_hybrid_report(result):
    print("=" * 55)
    print("           HYBRID SOIL ANALYSIS REPORT")
    print("=" * 55)

    print(f"Soil Type        : {result['Soil Type']}")
    print(f"Image Confidence: {result['Image Confidence'] * 100:.2f}%")

    print("\nNUTRIENT STATUS")
    print("-" * 55)
    print(
        f"N : {result['N Status']:<15} "
        f"Confidence: {result['N Confidence'] * 100:.2f}%"
    )
    print(
        f"P : {result['P Status']:<15} "
        f"Confidence: {result['P Confidence'] * 100:.2f}%"
    )
    print(
        f"K : {result['K Status']:<15} "
        f"Confidence: {result['K Confidence'] * 100:.2f}%"
    )

    print("\nOVERALL ASSESSMENT")
    print("-" * 55)
    print(f"Soil Health Score: {result['Soil Health Score']:.2f}/100")
    print(f"Assessment       : {result['Overall Assessment']}")

    print("=" * 55)

In [28]:
sample_500 = df.iloc[[500]]

result_500 = hybrid_soil_analysis(
    IMAGE_PATH,
    sample_500
)

display_hybrid_report(result_500)

           HYBRID SOIL ANALYSIS REPORT
Soil Type        : Alluvial_Soil
Image Confidence: 73.00%

NUTRIENT STATUS
-------------------------------------------------------
N : Deficient       Confidence: 96.34%
P : Not Deficient   Confidence: 99.06%
K : Deficient       Confidence: 91.14%

OVERALL ASSESSMENT
-------------------------------------------------------
Soil Health Score: 33.33/100
Assessment       : Two modeled nutrient deficiencies detected.


## Hybrid System Limitation

The current dataset does not provide verified pairings between individual
soil images and individual structured soil-test records.

Therefore, the hybrid demonstrations in this notebook use the image and
structured-data branches as separate inputs to verify the integration
pipeline. These examples demonstrate system functionality but should not
be interpreted as validation of a real physical soil sample.

Future validation should use paired image and structured soil-test
measurements collected from the same soil samples.

# Hybrid Soil Analysis — Final Summary

## System Architecture

The hybrid soil analysis system combines two complementary machine-learning
branches:

1. **Image Analysis**
   - Fine-tuned EfficientNetB0
   - Input: soil image
   - Output: predicted soil type and image confidence

2. **Structured Soil Analysis**
   - Gradient Boosting classifiers
   - Inputs: structured soil and environmental features
   - Outputs: N, P, and K deficiency status with prediction confidence

## Combination Logic

The two branches are combined at the assessment level rather than by directly
averaging probabilities, because they predict different targets.

- The image branch identifies the soil type.
- The structured branch estimates nutrient-deficiency status.
- The outputs are consolidated into a single soil-analysis report.

## Soil Health Score

A project-defined nutrient-status score is calculated from the three modeled
nutrients:

- 3/3 nutrients not deficient → 100
- 2/3 nutrients not deficient → 66.67
- 1/3 nutrient not deficient → 33.33
- 0/3 nutrients not deficient → 0

This score is a project-level indicator and is not a scientifically validated
soil-health index.

## Confidence Handling

Prediction confidence is reported separately for the image and each nutrient
prediction. This allows uncertain predictions to be identified rather than
hiding uncertainty inside the final score.

## Example

For the demonstrated sample:

- Soil type: Alluvial Soil
- Image confidence: approximately 73%
- N: Deficient
- P: Not Deficient
- K: Deficient
- Soil-health score: 33.33/100

## Important Limitation

The available dataset does not provide verified pairings between individual
soil images and individual structured soil-test records. Therefore, the hybrid
examples demonstrate technical integration rather than real-world validation
on matched physical soil samples.

Future work should use paired image and structured soil-test measurements from
the same soil samples and validate the complete hybrid system on those paired
observations.